In [ ]:
import numpy as np
import MODE_utilities as AS

In [ ]:
# ============================================================
# Global experiment variables
# ============================================================
# Full MODE budget
POPULATION = 20
OFFSPR = 10
GENS = 10
GAMMA = 1.0
NORMALIZE = True
SERIES_LENGTH = 500
MIN_CLUSTER_SIZE = 2
USE_DIVERGENCE = False
SPLIT = 0.80
MAX_ITER = 50
BARYCENTER_MAX_ITER = 50
D_STAR_TP = 1.8
# ASF weights
ASF_WEIGHTS_FULL = np.array([0.8, 0.2])
# Energy tracking
TRACK_ENERGY = True


In [ ]:
TYPE_OF_DATA = 'cpu'

split_data = AS.read_raspberry_pi_dataset(
    series_length=500, column=TYPE_OF_DATA
)

split_data = split_data[:60][:]

split_data


offline_data = split_data[:, :int(SPLIT*split_data.shape[1])]
online_data = split_data[:, int(SPLIT*split_data.shape[1]):]

In [ ]:

initial_output, initial_energy = AS.run_with_energy_tracking(
func=lambda: AS.ApplyingMODE_no_stability(
    data=offline_data,
    num_population=POPULATION,
    num_offsprings=OFFSPR,
    num_generations=GENS,
    gamma=GAMMA,
    max_iter=MAX_ITER,
    barycenter_max_iter=BARYCENTER_MAX_ITER,
    normalize=NORMALIZE,
    use_divergence=USE_DIVERGENCE,
    min_cluster_size=MIN_CLUSTER_SIZE,
    split=SPLIT,
    # random_state=RANDOM_STATE,
    D_star_tp=D_STAR_TP,
),
label="initial_MODE_H0",
enable=TRACK_ENERGY,
)

initial_results, initial_problem, initial_duration = initial_output

timing = initial_problem.timing

print("\nMODE timing breakdown")
print("=" * 50)

print(
    f"Total MODE:      "
    f"{timing['total_mode_sec']:.3f} sec"
)

print(
    f"Clustering:      "
    f"{timing['clustering_sec']:.3f} sec"
)

print(
    f"Training:        "
    f"{timing['training_sec']:.3f} sec"
)

print(
    f"Validation:      "
    f"{timing['validation_sec']:.3f} sec"
)

print(
    f"Stability:       "
    f"{timing['stability_sec']:.3f} sec"
)

print(
    f"Other/NSGA-II:   "
    f"{timing['other_sec']:.3f} sec"
)


print("\nExecution counts")
print("=" * 50)

print(
    "Clusterings:",
    timing["n_clusterings"]
)

print(
    "Training runs:",
    timing["n_trainings"]
)

print(
    "Validation runs:",
    timing["n_validations"]
)

print(
    "Stability evaluations:",
    timing["n_stability_evals"]
)

print(
    "Clustering cache hits:",
    timing["clustering_cache_hits"]
)

print("\nClustering time by K")

for k, duration in sorted(
    timing["clustering_by_k_sec"].items()
):
    print(
        f"K={k}: {duration:.3f} sec"
    )

initial_instance, initial_model, previous_chromosome, initial_info = AS.ApplyingASF_no_stability(
    results_of_tuning=initial_results,
    Problem_class=initial_problem,
    weights=ASF_WEIGHTS_FULL,
    return_full=True,
)

# initial_relative_hypervolumes, initial_hv_ideal, initial_hv_nadir = AS.collect_relative_hypervolume_history(
# initial_results,
# ideal=None,
# nadir=None,
# use_opt=True,
# )

# print("\nInitial relative hypervolumes [0, 1]:")
# print(initial_relative_hypervolumes)
print("\nConfiguration:")
print(previous_chromosome)
print("\nInitial energy [Wh]:")
print(initial_energy['energy_wh'])
print("\nInitial duration [sec]:")
print(initial_energy['duration_sec'])


In [ ]:
online_results = []

for series_id, ts in enumerate(online_data):

    res = AS.evaluate_one_individual_series(
        ML=initial_instance,
        series_raw=ts,
        sdtw_normalize=NORMALIZE,
        return_raw_scale=True,
    )

    online_results.append({
        "series_id": series_id,

        # Predictions / targets in original scale
        "predictions": res["preds_raw"],
        "targets": res["targets_raw"],

        # Prediction indices
        "time_index": res["prediction_time_index"],

        # Errors
        "rmse_normalized": res["rmse_used"],
        "rmse_raw": res["rmse_raw"],
        "nrmse_raw": res["nrmse_percent_raw"],
    })

# Save results

In [ ]:
import os
import pickle

os.makedirs("Experimental results", exist_ok=True)

with open(f"Experimental results/Raspberry_{TYPE_OF_DATA}_no_stab.pkl", "wb") as f:
    pickle.dump(online_results, f)

print("Online results _no_stab saved.")

In [ ]:

os.makedirs("Experimental results", exist_ok=True)

mode_summary = {
    # MODE timing
    "timing": initial_problem.timing,

    # Selected configuration
    "configuration": previous_chromosome,

    # Energy
    "energy": initial_energy,

    # Total duration measured by run_with_energy_tracking
    "initial_duration_sec": initial_duration,

    # MODE/Pymoo results
    "pareto_X": initial_results.X,
    "pareto_F": initial_results.F,

    # Generation history, if available
    "history": getattr(initial_results, "history", None),
}

with open(
    f"Experimental results/Raspberry_{TYPE_OF_DATA}_summary_no_stab.pkl",
    "wb"
) as f:
    pickle.dump(mode_summary, f)

print("MODE_no_stab summary saved.")